# Minimax Saddle Point, Extragradient, OGDA, SGA, and Trust-Region BOBYQA (C04 Topic 14)

**Citations:** Korpelevich (1976 Extragradient); Daskalakis et al. (2018 OGDA); Balduzzi et al. (2018 SGA); Powell (2009 BOBYQA); Nelder & Mead (1965 Simplex); Heusel et al. (2017 TTUR).

### Purpose & Topic Bridges
Single-objective optimization minimizes a scalar potential function where gradient descent naturally descends along conservative vector fields (zero rotation). In stark contrast, multi-agent adversarial games—including Generative Adversarial Networks (GANs), robust minimax control, actor-critic reinforcement learning, and primal-dual formulations—optimize conflicting objectives $\min_x \max_y f(x, y)$. The game vector field contains a non-conservative **rotational (Hamiltonian) component**.

Under standard **Simultaneous Gradient Descent-Ascent (GDA)**, rotational game dynamics force iterates to step tangentially along circular contours, causing catastrophic exponential divergence to $\infty$. **Korpelevich Extragradient (`EG`)** evaluates a lookahead gradient predictor to impart negative rotational friction, proving linear geometric contraction to the Nash equilibrium. **Optimistic GDA (`OGDA`)** achieves this contraction using past gradient momentum with only a single gradient evaluation per step. **Symplectic Gradient Adjustment (`SGA`)** decomposes the game Jacobian to directly quench the rotational component. For derivative-free black-box landscapes, **Nelder-Mead Simplex** and **Powell's BOBYQA** provide gradient-free trust-region optimization.

**Decision Boundary:** Use **Extragradient** or **OGDA** whenever optimizing adversarial, zero-sum, or minimax objectives. Use **SGA** when game dynamics exhibit severe rotational oscillations. Use **Nelder-Mead / BOBYQA** for low-dimensional black-box physics simulators without autograd. When NOT to use: Never use naive simultaneous **GDA** without optimism or extragradient corrections on bilinear/saddle-point games; GDA provably diverges for any positive step size $\eta > 0$.

## 2. Mathematical Foundations & Governing Equations

1. **Bilinear Game & Vector Field Jacobian:** For $\min_x \max_y x^T A y$, vector field $F(z) = [Ay, -A^T x]^T$ has Jacobian $J_F = [[0, A], [-A^T, 0]]$ with purely imaginary eigenvalues $\pm i \sigma$.
2. **Simultaneous GDA Divergence:** Update $z_{t+1} = (I - \eta J_F) z_t$ has eigenvalues with modulus $|\lambda| = \sqrt{1 + \eta^2 \sigma^2} > 1$, diverging as $\|z_t\|_2 = (1 + \eta^2 \sigma^2)^{t/2} \|z_0\|_2 \to \infty$.
3. **Korpelevich Extragradient Contraction:**
$$\tilde{z}_t = z_t - \eta F(z_t), \quad z_{t+1} = z_t - \eta F(\tilde{z}_t) \implies |\lambda_{\text{EG}}|^2 = 1 - \eta^2 \sigma^2 (1 - \eta^2 \sigma^2) < 1$$
4. **Optimistic GDA (OGDA):** Approximates Extragradient with past momentum in 1 gradient evaluation:
$$z_{t+1} = z_t - 2\eta F(z_t) + \eta F(z_{t-1})$$
5. **Symplectic Gradient Adjustment (SGA):** With rotation $A = \frac{1}{2}(J_F - J_F^T)$:
$$F_{\text{SGA}} = F + \lambda A^T F = \begin{bmatrix} Ay + \lambda A A^T x \\ -A^T x + \lambda A^T A y \end{bmatrix}$$
6. **Nelder-Mead Simplex Reflection:** Centroid $\bar{x} = \frac{1}{2}(x_0 + x_1)$, reflected point $x_r = \bar{x} + \alpha (\bar{x} - x_2)$.

In [ ]:
import sys
from pathlib import Path
import numpy as np, torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/04_training_methods_and_optimisers/02_constrained_pareto_and_blackbox_optimisation/07_minimax_saddle_point_extragradient_ogda_sga_and_trust_region_bobyqa.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
def step_extragradient(x, y, A, eta_x=0.1, eta_y=0.1):
    gx, gy = torch.matmul(A, y), torch.matmul(A.T, x)
    x_tilde, y_tilde = x - eta_x * gx, y + eta_y * gy
    gx_tilde, gy_tilde = torch.matmul(A, y_tilde), torch.matmul(A.T, x_tilde)
    return x - eta_x * gx_tilde, y + eta_y * gy_tilde

def step_ogda(x, y, x_prev, y_prev, A, eta_x=0.1, eta_y=0.1):
    gx, gy = torch.matmul(A, y), torch.matmul(A.T, x)
    gxp, gyp = torch.matmul(A, y_prev), torch.matmul(A.T, x_prev)
    return x - 2.0 * eta_x * gx + eta_x * gxp, y + 2.0 * eta_y * gy - eta_y * gyp

def step_sga(x, y, A, lambda_sga=0.5, eta_x=0.1, eta_y=0.1):
    AAT_x, ATA_y = torch.matmul(A, torch.matmul(A.T, x)), torch.matmul(A.T, torch.matmul(A, y))
    gx, gy = torch.matmul(A, y), torch.matmul(A.T, x)
    return x - eta_x * (gx + lambda_sga * AAT_x), y + eta_y * (gy - lambda_sga * ATA_y)

def nelder_mead_reflect(x_best, x_second, x_worst, alpha=1.0):
    centroid = 0.5 * (x_best + x_second)
    return centroid + alpha * (centroid - x_worst)

class LinearAnchor(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(out_f, in_f))
        self.bias = nn.Parameter(torch.zeros(out_f))
    def forward(self, x):
        return torch.matmul(x, self.weight.T) + self.bias

print('MinimaxUnifiedModule routines defined successfully.')

In [ ]:
# 1. Extragradient Invariant
x_eg, y_eg = step_extragradient(weights['state_x'], weights['state_y'], weights['matrix_A'], 0.1, 0.1)
err_eg = max(float(torch.max(torch.abs(x_eg - weights['golden_x_eg']))), float(torch.max(torch.abs(y_eg - weights['golden_y_eg']))))

# 2. OGDA Invariant
x_ogda, y_ogda = step_ogda(weights['state_x'], weights['state_y'], weights['state_x_prev'], weights['state_y_prev'], weights['matrix_A'], 0.1, 0.1)
err_ogda = max(float(torch.max(torch.abs(x_ogda - weights['golden_x_ogda']))), float(torch.max(torch.abs(y_ogda - weights['golden_y_ogda']))))

# 3. SGA Invariant
x_sga, y_sga = step_sga(weights['state_x'], weights['state_y'], weights['matrix_A'], 0.5, 0.1, 0.1)
err_sga = max(float(torch.max(torch.abs(x_sga - weights['golden_x_sga']))), float(torch.max(torch.abs(y_sga - weights['golden_y_sga']))))

# 4. Nelder-Mead Reflection Invariant
x_refl = nelder_mead_reflect(weights['x_best'], weights['x_second'], weights['x_worst'], 1.0)
err_refl = float(torch.max(torch.abs(x_refl - weights['golden_x_reflect'])))

# 5. Linear Anchor Invariant
anchor = LinearAnchor(8, 4)
anchor.weight.data.copy_(weights['anchor_w'].T)
anchor.bias.data.copy_(weights['anchor_b'])
anchor_out = anchor(weights['anchor_x']).detach()
err_anchor = float(torch.max(torch.abs(anchor_out - weights['golden_anchor_out'])))

for name, err in [('Extragradient', err_eg), ('OGDA', err_ogda), ('SGA', err_sga), ('Nelder-Mead', err_refl), ('Anchor', err_anchor)]:
    assert err < 1e-5, f'{name} invariant failed: {err:.2e}'
print(f'All 5 invariants verified! EG: {err_eg:.2e}, OGDA: {err_ogda:.2e}, SGA: {err_sga:.2e}, Nelder-Mead: {err_refl:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: anchor(weights['anchor_x']),
    'Extragradient Step': lambda: step_extragradient(weights['state_x'], weights['state_y'], weights['matrix_A']),
    'OGDA Step': lambda: step_ogda(weights['state_x'], weights['state_y'], weights['state_x_prev'], weights['state_y_prev'], weights['matrix_A']),
    'SGA Step': lambda: step_sga(weights['state_x'], weights['state_y'], weights['matrix_A']),
    'Nelder-Mead Reflect': lambda: nelder_mead_reflect(weights['x_best'], weights['x_second'], weights['x_worst']),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=8)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Active Failure Mode: Catastrophic Rotational Divergence in Simultaneous GDA vs Extragradient Contraction
# Bilinear game: min_x max_y x * y. Initial state: (x0, y0) = (1.0, 1.0), Nash equilibrium: (0, 0).
eta = 0.5
# Run 25 steps of Simultaneous GDA: x_{t+1} = x_t - eta * y_t, y_{t+1} = y_t + eta * x_t
xg, yg = torch.tensor(1.0), torch.tensor(1.0)
for _ in range(25):
    new_x = xg - eta * yg
    new_y = yg + eta * xg
    xg, yg = new_x, new_y
norm_gda = torch.sqrt(xg**2 + yg**2).item()

# Run 25 steps of Korpelevich Extragradient: lookahead dampens rotation and contracts to origin
xe, ye = torch.tensor(1.0), torch.tensor(1.0)
for _ in range(25):
    xt, yt = xe - eta * ye, ye + eta * xe
    xe, ye = xe - eta * yt, ye + eta * xt
norm_eg = torch.sqrt(xe**2 + ye**2).item()

print(f'[FAILURE DEMONSTRATED] Simultaneous GDA exploded outward from norm 1.414 to {norm_gda:.2f}!')
print(f'[CORRECT] Extragradient contracted inward to Nash equilibrium: norm {norm_eg:.4f}!')
assert norm_gda > 10.0 and norm_eg < 0.15, 'Rotational divergence vs contraction not cleanly reproduced.'
print('[REMEDY] Always use Extragradient, OGDA, or SGA on minimax objectives to impart negative rotational friction.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Game Dynamics Differ Fundamentally from Minimization:** Minimax vector fields possess rotational Hamiltonian vorticity that causes standard gradient descent to spiral outward.
2. **Lookahead Imparts Negative Rotational Friction:** Korpelevich Extragradient and OGDA dampen rotations by evaluating lookahead gradients or past momentum, guaranteeing linear geometric contraction to Nash equilibria.
3. **Milestone Achieved — Category 04 Complete:** This concludes Category 04 (*Training Methods & Optimizers*), spanning 20 comprehensive topics across First-Order, Second-Order, Schedule-Free, Constrained, Pareto, Blackbox, SAM, and Minimax optimization.

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 06_sharpness_aware_swad_and_weight_averaging_dynamics](file:///c:/Users/Liam/Code/AI-Examples/notebooks/04_training_methods_and_optimisers/02_constrained_pareto_and_blackbox_optimisation/06_sharpness_aware_swad_and_weight_averaging_dynamics.ipynb) | [Next: Category 05 — Architectures & Sequence Models](file:///c:/Users/Liam/Code/AI-Examples/notebooks/05_architectures/)